# Split-Hemisphere Atlas Heatmaps

Coronal-slice grid with independent data on the left and right hemispheres,
each fed from its **own** CSV/region/value column. Built to support plotting
two genuinely different metrics side by side -- e.g. average intensity on
one hemisphere, significance (or fold-change, or anything else) on the
other -- where the two source tables may not cover the same set of regions.

**Hemisphere selection (`HEMISPHERE` in Section 1).**

Set `HEMISPHERE = "left"` or `"right"` to render a single hemisphere; the
default `"both"` keeps the original split-hemisphere behavior unchanged. When a
single side is selected:

- only that side's data is drawn, and the frame is cropped to that hemisphere
  rather than leaving the other half blank
- panels are re-proportioned so a half-brain doesn't sit in a square box with
  whitespace either side (`PANEL_SIZE` now sets the *larger* dimension)
- one colorbar instead of two, and one label instead of two
- the midline is drawn as the cut edge of the section
- `SINGLE_SOURCE` chooses which of the two configured data blocks fills it, so
  you can show, say, the `RIGHT_VALUE_COL` metric on a left-facing hemisphere
  without shuffling the config

**A caveat worth putting in the figure legend.** Region values in these tables
are whole-structure, not per-hemisphere — the pipeline quantifies a region, not
a region-within-a-side. Showing one hemisphere is therefore a display choice
about redundancy, not a lateralized measurement. Caption it as a representative
hemisphere; don't let it read as though left and right were measured separately.

**What changed in the previous edit, and why:**

- **Two independent input files.** `LEFT_CSV_PATH`/`RIGHT_CSV_PATH` (with
  their own region/value column names) replace the old single
  `CSV_PATH` + two-column setup. If you want the old single-file behavior
  back, just point both paths at the same CSV.
- **Asymmetric region coverage is now the expected case, not a warning
  sign.** A significance table will usually cover far fewer regions than an
  intensity-averages table -- regions present on one side only render
  `MISSING_COLOR` on the other, and the coverage-diff printout is reworded
  to say so rather than read as an error.
- **`SHARE_COLOR_SCALE` now defaults to `False`.** The old default (`True`)
  pooled both sides into one shared color range -- correct when both sides
  are the same metric (e.g. light vs. dark density), but wrong once the two
  sides are different metrics with different units/scales (e.g. density vs.
  -log10(q)): pooling would compute nonsense limits. Flip it back to `True`
  only when both sides really are commensurate.
- **Midline is now solid and heavier** (`MIDLINE_LW`, `MIDLINE_COLOR` in
  config) instead of a thin dashed line.
- **Each hemisphere is now labeled directly above its own half of every
  panel** (`LEFT_LABEL`/`RIGHT_LABEL` in config, default to the value column
  names) -- not just once in a shared colorbar label.
- **New: an ontology-color reference grid** (Section 6) -- the same coronal
  grid, but every region filled with the Allen atlas's own default color
  instead of your data. Meant to sit next to the data grid as a legend for
  "what region is this," since a coronal heatmap alone doesn't tell you that.
- **New: a duplicate-region guard in `build_values`.** If your value CSV has
  more than one row per region (e.g. an un-filtered multi-contrast LMM output
  table, where each region appears once per contrast), the old code would
  silently keep whichever row happened to come last. It now warns explicitly
  and tells you which regions were affected, since this is a real risk if you
  feed this notebook straight from the per-region LMM's output CSV without
  filtering to one contrast first.


---

## Hygiene pass (this version)

No plotting behavior changed. Config only:

- **Paths.** Both sources were hardcoded to
  `~/Desktop/cFos_analysis/Python/FINAL/figure_data/...`, which does not exist on
  another machine and is not where the PLSC notebook writes. Replaced with
  `FIGURE_DATA_DIR` + `RUN_TAG` + a `PANEL` selector, so switching between the
  5H (light) and 5I (three-way) exports is one line. A missing file now raises
  with the path and the likely cause instead of failing inside `read_csv`.
- **`LEFT_LABEL` was `" "`, not `None`.** The `LEFT_LABEL or LEFT_VALUE_COL`
  fallback treats a single space as truthy, so the left colorbar rendered blank
  rather than defaulting to the value-column name. Both labels are now `None`.
- **`SHARE_COLOR_SCALE`.** The comment read "Defaults to False now" while the
  value was `True`. The value is right for the current config (both sides read
  one metric); the comment was corrected to match.
- **`RIGHT_NEG_RGB` differed from `LEFT_NEG_RGB`.** With a shared numeric scale
  and one metric on both halves, equal values were rendering in different
  colours. Matched.
- **Atlas resolution.** Noted that `allen_mouse_10um` here is a rendering choice
  while the analysis notebooks use 25um; the ontology is the same, so acronyms
  match, but Methods should state both.

### Caveat for the figure legend, unchanged and worth repeating

Region values are **whole-structure**, not per-hemisphere — the pipeline
quantifies a region, not a region-within-a-side. Showing one hemisphere is a
display choice about redundancy, not a lateralized measurement. Caption it as a
representative hemisphere.

If the mapped column is `BSR`: that is a **stability** measure, uncorrected
across regions. "Stable contributors", never "significantly different".

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm, to_rgb
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle
from collections import defaultdict
from pathlib import Path

import brainglobe_heatmap as bgh
from brainglobe_atlasapi import BrainGlobeAtlas


## 1. Config

In [ ]:
# ── Where notebook 04 writes its panel exports ───────────────────────────────
# ---------------------------------------------------------------------------
# Paths -- repo-relative, so this notebook runs wherever the repository is
# checked out. PROJECT_ROOT is found by walking up from the working directory
# until a folder containing `data/` appears. If you keep the data outside the
# repository, set PROJECT_ROOT by hand and everything below follows.
# ---------------------------------------------------------------------------
def find_project_root(markers=("data", "notebooks")):
    """Nearest ancestor directory containing all of `markers`. Requiring both
    `data/` and `notebooks/` means a stray `data/` folder somewhere on the path
    cannot be mistaken for the repository root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError(
        f"No ancestor of {here} contains {list(markers)}. Set PROJECT_ROOT by hand."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

# Must match notebook 04's FIGURE_DATA_DIR.
FIGURE_DATA_DIR = RESULTS_DIR / "figure_data"
RUN_TAG         = "cellmean_gainoff"       # must match the PLSC notebook's RUN_TAG

# ### CHOICE ### which exported panel to render.
#   "5I_three_way" -> panel5I_three_way_heatmap_<RUN_TAG>.csv
#   "5H_light"     -> panel5H_light_heatmap_<RUN_TAG>.csv
# The PLSC notebook writes both, each carrying estimate / se / t / p / q /
# salience / unit_salience / BSR, so VALUE_COL picks which column is mapped.
PANEL     = "5I_three_way"
VALUE_COL = "BSR"          # "BSR" (stability) or "estimate" (effect size)

PANEL_CSV = FIGURE_DATA_DIR / f"panel{PANEL}_heatmap_{RUN_TAG}.csv"

# ### NOTE ###
# BSR is a STABILITY measure, not a significance test, and it is uncorrected
# across regions. If this figure maps BSR, the legend must say "stable
# contributors" -- never "significantly different". Map "estimate" instead if the
# figure is meant to show effect size.

# ── Data: LEFT and RIGHT are fully independent sources ───────────────────────
# Point both at the same file/column for a single-metric plot (the current
# setup). They exist as separate blocks so two different metrics can be shown
# side by side -- e.g. intensity on one hemisphere, significance on the other --
# with no requirement that they cover the same regions.
LEFT_CSV_PATH   = PANEL_CSV
LEFT_REGION_COL = "region"
LEFT_VALUE_COL  = VALUE_COL
LEFT_LABEL      = None    # None -> defaults to LEFT_VALUE_COL; set e.g. "Dark"

RIGHT_CSV_PATH   = PANEL_CSV
RIGHT_REGION_COL = "region"
RIGHT_VALUE_COL  = VALUE_COL
RIGHT_LABEL      = None   # None -> defaults to RIGHT_VALUE_COL; set e.g. "Light"

# ### FIXED ### LEFT_LABEL was " " (a single space), not None. The
# `LEFT_LABEL or LEFT_VALUE_COL` fallback at the bottom of this cell treats " "
# as truthy, so the left colorbar silently rendered with a blank label rather
# than falling back to the value-column name. Use None to mean "default".

# ### NOTE ###
# If RIGHT_CSV_PATH points at a per-region LMM contrast table (e.g. the
# per-region LMM notebook's OUT_CSV), that file has ONE ROW PER REGION *PER
# CONTRAST* -- filter it down to a single contrast before loading here, e.g.:
#   df = pd.read_csv(OUT_CSV)
#   df[df.contrast == "light_vs_dark | control | M"].to_csv("sig_for_plot.csv")
# build_values() below will warn (not silently misbehave) if it finds
# duplicate region entries, but filtering upstream is the correct fix.

# ── Hemisphere selection ─────────────────────────────────────────────────────
# "both"  -> original split-hemisphere plot (left source | right source)
# "left"  -> single hemisphere, cropped to the left half of the image
# "right" -> single hemisphere, cropped to the right half of the image
#
# ### NOTE ###
# Region values here are whole-structure, not per-hemisphere. Plotting one side
# is a statement about redundancy, not about lateralization. Say "representative
# hemisphere" in the legend.
HEMISPHERE = "left"

# Which data block fills the panel when HEMISPHERE is not "both".
# None -> use the block matching HEMISPHERE (left side uses LEFT_*, etc.).
# Set explicitly to show e.g. the RIGHT_VALUE_COL metric on a left hemisphere.
SINGLE_SOURCE = None            # None | "left" | "right"

# ── Atlas ────────────────────────────────────────────────────────────────────
# ### NOTE ### 10um here is a RENDERING choice -- finer boundaries in the figure.
# The analysis notebooks use allen_mouse_25um. The ontology is identical between
# them, so acronyms match; only the annotation raster differs. Nothing about the
# statistics depends on this, but Methods should state both resolutions rather
# than implying one was used throughout.
ATLAS_NAME = "allen_mouse_10um"

# ── Hierarchy ────────────────────────────────────────────────────────────────
# Depth = len(structure_id_path).
#   root=1 > grey=2 > CH=3 > CTX=4 > CTXpl=5 > Isocortex=6 > MO=7 > MOp=8 > MOp1=9
# MAX_DEPTH=8 keeps MOp / VISp / SSp-bfd and drops their layer subdivisions.
MAX_DEPTH      = 8
DROP_ANCESTORS = True

# ── Anatomical overlays: drawn as anatomy, UNDER the data, never data-colored ─
# Top-level acronyms only -- each subsumes its children (opt/och subset of
# fiber tracts; VL/V3/V4/AQ subset of VS). Low zorder so region fills sit on top.
OVERLAY_SPEC = {
    "fiber tracts": dict(face="#d9d9d9", edge="#9e9e9e", lw=0.5, alpha=1.0, zorder=1),
    "VS":           dict(face="#595959", edge="#3d3d3d", lw=0.5, alpha=1.0, zorder=2),
}
OVERLAY_FALLBACKS = ["fiber_tracts", "fibertracts", "ft", "VS", "V", "VL", "V3", "AQ"]

# ── Slices (µm from anterior pole) ───────────────────────────────────────────
CORONAL_POSITIONS = [2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 6500,
                      7000, 7500, 8000, 8500, 9000, 9500, 10000, 10500, 11000]

# ── Grid layout ──────────────────────────────────────────────────────────────
GRID_ROWS, GRID_COLS = 3, 6
PANEL_SIZE   = 2.6      # inches: sets the LARGER panel dimension; the shorter
                        # axis shrinks to match the data's aspect, so a single
                        # hemisphere gets a tall narrow panel instead of a square
                        # one with whitespace on both sides.

# ### CHOICE ###
# The original -0.4 pulled rows together to claw back the vertical whitespace a
# square frame left above and below each slice. Cropping to one hemisphere
# removes most of that slack, so a single-hemisphere plot needs a much gentler
# value or the rows collide. None -> pick automatically from HEMISPHERE.
GRID_HSPACE  = None     # None | float
SCALEBAR_MM  = 1.0
PANEL_PAD_UM = 100      # margin around the widest slice
BREGMA_UM    = None     # e.g. 5400 to title panels bregma-relative; None = raw µm

# ── Midline ──────────────────────────────────────────────────────────────────
# With HEMISPHERE="both" this divides the two data sources. With a single
# hemisphere it is the cut edge of the section; set DRAW_MIDLINE=False to hide it.
MIDLINE_COLOR = "black"
MIDLINE_LW    = 1.4      # was 0.6 dashed -- now solid and heavier, per request
MIDLINE_ALPHA = 1.0
DRAW_MIDLINE  = True

# ── Per-hemisphere labels, drawn above each panel's own slice ────────────────
LABEL_EACH_PANEL = True
LABEL_FONTSIZE   = 8
LABEL_Y_PAD_UM   = 150    # gap between the top of the anatomy and the label

# ── Colors: '#RRGGBB', (r,g,b) 0-255, or (r,g,b) 0-1 ─────────────────────────
LEFT_RGB      = (203, 212, 72)
RIGHT_RGB     = (203, 212, 72)
LEFT_NEG_RGB  = (92, 162, 216)      # None = white->color ramp
# ### HYGIENE ### was (153, 153, 51), a different negative arm from the left
# side. With SHARE_COLOR_SCALE=True and both blocks reading one metric, the two
# halves were on a shared numeric scale but different colour ramps, so equal
# values rendered as different colours. Matched to LEFT_NEG_RGB.
RIGHT_NEG_RGB = (92, 162, 216)

# ── Scale ────────────────────────────────────────────────────────────────────
# Delta columns  -> CENTER_AT_ZERO=True, VMIN_FIXED=None, NEG_RGB set.
# Raw densities  -> CENTER_AT_ZERO=False, VMIN_FIXED=0, NEG_RGB=None.
CENTER_AT_ZERO    = True

# ### CHOICE ###
# SHARE_COLOR_SCALE=True pools left+right values into one shared color range.
# That is correct ONLY when both sides are the same metric -- which is the case
# in the current config, since both blocks read the same column of the same file.
# If the two sides are ever pointed at different metrics with different units
# (density vs. -log10(q), say), set this to False: a shared pool would compute
# limits that mean nothing for either side.
#
# ### FIXED ### the comment here used to read "Defaults to False now" while the
# value was True, so the notebook documented the opposite of what it did.
SHARE_COLOR_SCALE = True
SYMMETRIC_LIMITS  = True    # +-max(|vmin|,|vmax|); equal saturation per arm, per side
VMAX_PERCENTILE   = 97
VMAX_FIXED        = 3
VMIN_FIXED        = -3

MISSING_COLOR = "#e8e8e8"
OUTPUT_DIR    = RESULTS_DIR / "figures" / "atlas"
SAVE_FIGURES  = True

LEFT_LABEL  = LEFT_LABEL  or LEFT_VALUE_COL
RIGHT_LABEL = RIGHT_LABEL or RIGHT_VALUE_COL

# ── Resolve hemisphere settings ──────────────────────────────────────────────
HEMISPHERE = HEMISPHERE.lower()
assert HEMISPHERE in ("both", "left", "right"), \
    f'HEMISPHERE must be "both", "left" or "right", got {HEMISPHERE!r}'

SINGLE = HEMISPHERE != "both"
SOURCE = (SINGLE_SOURCE or HEMISPHERE) if SINGLE else None
if SINGLE:
    assert SOURCE in ("left", "right"), \
        f'SINGLE_SOURCE must be None, "left" or "right", got {SINGLE_SOURCE!r}'

if GRID_HSPACE is None:
    # Per-panel labels need vertical room; without them a single hemisphere can
    # be packed tight.
    GRID_HSPACE = (0.12 if LABEL_EACH_PANEL else 0.02) if SINGLE else -0.4

if SINGLE:
    print(f"HEMISPHERE={HEMISPHERE!r}: drawing the {HEMISPHERE} half only, "
          f"filled from the {SOURCE.upper()}_* data block.")
    if LABEL_EACH_PANEL:
        print("  note: LABEL_EACH_PANEL repeats the same metric name on every "
              "panel. With one hemisphere there is only one metric and the "
              "colorbar already names it -- consider LABEL_EACH_PANEL=False.")
else:
    print("HEMISPHERE='both': split-hemisphere plot (original behavior).")

# ### HYGIENE ### fail here, with the path, rather than inside a read_csv
# traceback three cells later.
for _side, _p in [("LEFT", LEFT_CSV_PATH), ("RIGHT", RIGHT_CSV_PATH)]:
    if not Path(_p).exists():
        raise FileNotFoundError(
            f"{_side}_CSV_PATH does not exist:\n  {_p}\n"
            f"Set FIGURE_DATA_DIR to the PLSC notebook's FIGURE_DATA_DIR, and check "
            f"RUN_TAG={RUN_TAG!r} matches the tag that notebook printed.")
print(f"\nrendering {PANEL}  column {VALUE_COL!r}  <- {PANEL_CSV}")

## 2. Color/geometry helpers

In [ ]:
def _to_rgb(c):
    if isinstance(c, str):
        return to_rgb(c)
    c = tuple(c)
    return tuple(v / 255 for v in c) if max(c) > 1 else c


def make_cmap(pos_rgb, neg_rgb=None, name="custom"):
    if neg_rgb is None:
        return LinearSegmentedColormap.from_list(name, ["white", _to_rgb(pos_rgb)])
    return LinearSegmentedColormap.from_list(
        name, [_to_rgb(neg_rgb), "white", _to_rgb(pos_rgb)]
    )


def make_norm(vmin, vmax):
    if CENTER_AT_ZERO and vmin < 0 < vmax:
        return TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    return Normalize(vmin=vmin, vmax=vmax)


def _poly_area(c):
    x, y = c[:, 0], c[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


## 3. Load atlas, resolve overlays, load + process both value tables

In [ ]:
atlas = BrainGlobeAtlas(ATLAS_NAME)


# ── Overlay resolution ───────────────────────────────────────────────────────
def resolve_overlays():
    ok, missing = {}, []
    for acr, style in OVERLAY_SPEC.items():
        try:
            atlas.structures[acr]
            ok[acr] = style
            print(f"overlay OK: {acr!r:16s} -> {atlas.structures[acr]['name']}")
        except KeyError:
            missing.append(acr)

    if missing:
        print(f"\nNOT in {ATLAS_NAME}: {missing}")
        print("  probing alternatives --")
        for acr in OVERLAY_FALLBACKS:
            try:
                print(f"    {acr!r} exists -> {atlas.structures[acr]['name']}")
            except KeyError:
                pass
        print("  put the correct key in OVERLAY_SPEC and re-run.\n")
    return ok


OVERLAYS = resolve_overlays()

OVERLAY_FAMILY = set(OVERLAYS)
for acr in OVERLAYS:
    try:
        OVERLAY_FAMILY |= set(atlas.get_structure_descendants(acr))
    except KeyError:
        pass
print(f"overlay family: {len(OVERLAY_FAMILY)} acronyms excluded from data fills")


# ── Load both value tables independently ─────────────────────────────────────
df_left  = pd.read_csv(LEFT_CSV_PATH)
df_right = pd.read_csv(RIGHT_CSV_PATH)
print(f"\nLEFT  ({LEFT_CSV_PATH}) columns:  {list(df_left.columns)}")
print(f"RIGHT ({RIGHT_CSV_PATH}) columns: {list(df_right.columns)}")


def build_values(df, region_col, value_col, label=""):
    """region -> value dict, dropping non-numeric/NaN entries.

    ### CHOICE ###
    Warns (rather than silently keeping an arbitrary row) if region_col has
    duplicate entries -- e.g. a multi-contrast LMM output table that hasn't
    been filtered to one contrast yet. Dict construction would otherwise just
    keep whichever duplicate came last, with no indication anything was
    dropped.
    """
    dup_mask = df[region_col].duplicated(keep=False)
    if dup_mask.any():
        dup_regions = sorted(df.loc[dup_mask, region_col].unique())
        print(f"  !! [{label}] {region_col} has duplicate entries for "
              f"{len(dup_regions)} region(s) -- only the LAST row per region "
              f"will be used: {dup_regions[:10]}{'...' if len(dup_regions) > 10 else ''}")
        print(f"     If this came from a multi-contrast table, filter to one "
              f"contrast before loading it here.")

    s = pd.to_numeric(df[value_col], errors="coerce")
    return {str(k): float(x) for k, x in zip(df[region_col], s) if pd.notna(x)}


# ── Hierarchy helpers ────────────────────────────────────────────────────────
def depth(acr):
    return len(atlas.structures[acr]["structure_id_path"])


def depth_histogram(vals, label=""):
    by_depth = defaultdict(list)
    for acr in vals:
        try:
            by_depth[depth(acr)].append(acr)
        except KeyError:
            pass
    print(f"\nhierarchy depths in {label}:")
    for d in sorted(by_depth):
        print(f"  depth {d}: {len(by_depth[d]):3d}  e.g. {sorted(by_depth[d])[:8]}")
    return by_depth


def collapse_to_depth(vals, max_depth, label=""):
    """Drop regions deeper than max_depth. Parents already carry their own
    value in the table, so deep regions are discarded, not averaged."""
    kept, rolled, orphaned = {}, set(), set()
    for acr, v in vals.items():
        try:
            d = depth(acr)
        except KeyError:
            continue
        if d <= max_depth:
            kept[acr] = v
            continue
        path = atlas.structures[acr]["structure_id_path"]
        anc = [atlas.structures[i]["acronym"] for i in path[:-1]]
        hit = next((a for a in reversed(anc)
                    if depth(a) <= max_depth and a in vals), None)
        (rolled if hit else orphaned).add(acr)

    print(f"  [{label}] depth<={max_depth}: kept {len(kept)}, "
          f"rolled up {len(rolled)}, orphaned {len(orphaned)}")
    if orphaned:
        print(f"    NO parent in table (dropped entirely): {sorted(orphaned)[:15]}")
    return kept


def drop_ancestors(vals, label=""):
    """Remove any region that still has a descendant present -- otherwise the
    parent polygon (CTX, CH, BS...) paints over its own children."""
    present, drop, unknown = set(vals), set(), []
    for acr in present:
        try:
            if set(atlas.get_structure_descendants(acr)) & present:
                drop.add(acr)
        except KeyError:
            unknown.append(acr)
    if unknown:
        print(f"  [{label}] not in atlas ({len(unknown)}): {sorted(unknown)[:10]}")
    print(f"  [{label}] dropped {len(drop)} ancestors, kept {len(present) - len(drop)}")
    return {k: v for k, v in vals.items() if k not in drop}


# ── Build values, independently per side ─────────────────────────────────────
values_left  = build_values(df_left,  LEFT_REGION_COL,  LEFT_VALUE_COL,  LEFT_LABEL)
values_right = build_values(df_right, RIGHT_REGION_COL, RIGHT_VALUE_COL, RIGHT_LABEL)

depth_histogram(values_left,  LEFT_LABEL)
depth_histogram(values_right, RIGHT_LABEL)

values_left  = collapse_to_depth(values_left,  MAX_DEPTH, LEFT_LABEL)
values_right = collapse_to_depth(values_right, MAX_DEPTH, RIGHT_LABEL)

if DROP_ANCESTORS:
    values_left  = drop_ancestors(values_left,  LEFT_LABEL)
    values_right = drop_ancestors(values_right, RIGHT_LABEL)

for dvals in (values_left, values_right):
    for acr in OVERLAY_FAMILY:
        dvals.pop(acr, None)

print(f"\ndata regions: {LEFT_LABEL}={len(values_left)}  {RIGHT_LABEL}={len(values_right)}")

# ### NOTE ###
# Asymmetric coverage here is EXPECTED, not an error, whenever the two sides
# are different kinds of data (e.g. a sparse significance table on one side,
# a dense intensity-average table on the other). Regions present on only one
# side render MISSING_COLOR on the other side -- this is just telling you
# which regions that applies to.
only_left  = set(values_left)  - set(values_right)
only_right = set(values_right) - set(values_left)
if only_left or only_right:
    print("\nRegions present on only one side (render MISSING_COLOR on the other):")
    print(f"  only in {LEFT_LABEL}  ({len(only_left)}): {sorted(only_left)}")
    print(f"  only in {RIGHT_LABEL} ({len(only_right)}): {sorted(only_right)}")


# ── Limits ───────────────────────────────────────────────────────────────────
def limits(vals, pool):
    src = pool if SHARE_COLOR_SCALE else np.array(list(vals.values()))
    if VMAX_FIXED is not None:
        vmax = VMAX_FIXED
    else:
        vmax = float(np.percentile(src, VMAX_PERCENTILE))
    if VMIN_FIXED is not None:
        vmin = VMIN_FIXED
    else:
        vmin = float(np.percentile(src, 100 - VMAX_PERCENTILE))
    if SYMMETRIC_LIMITS and vmin < 0 < vmax:
        m = max(abs(vmin), abs(vmax))
        vmin, vmax = -m, m
    return vmin, vmax


# ### CHOICE ###
# When only one hemisphere is drawn, the other side's values are not on the
# figure at all, so pooling them into a shared colour range would set limits
# from numbers the reader never sees. Pool over the shown side only.
if SINGLE:
    shown_vals = values_left if SOURCE == "left" else values_right
    pool = np.array(list(shown_vals.values()))
else:
    pool = np.array(list(values_left.values()) + list(values_right.values()))

LEFT_VMIN,  LEFT_VMAX  = limits(values_left,  pool)
RIGHT_VMIN, RIGHT_VMAX = limits(values_right, pool)
print(f"\nscale  {LEFT_LABEL}: [{LEFT_VMIN:.3g}, {LEFT_VMAX:.3g}]   "
      f"{RIGHT_LABEL}: [{RIGHT_VMIN:.3g}, {RIGHT_VMAX:.3g}]"
      f"  (shared pool: {SHARE_COLOR_SCALE})")
if SINGLE:
    print(f"       single hemisphere -> limits computed from the "
          f"{SOURCE.upper()} values only ({len(pool)} regions)")

LEFT_CMAP  = make_cmap(LEFT_RGB,  LEFT_NEG_RGB,  "left")
RIGHT_CMAP = make_cmap(RIGHT_RGB, RIGHT_NEG_RGB, "right")
LEFT_NORM  = make_norm(LEFT_VMIN,  LEFT_VMAX)
RIGHT_NORM = make_norm(RIGHT_VMIN, RIGHT_VMAX)


## 4. Slice projection and drawing

In [ ]:
_PROJ_CACHE = {}


def _get_projected(position, orientation="frontal"):
    key = (position, orientation)
    if key in _PROJ_CACHE:
        return _PROJ_CACHE[key]
    # Only request meshes for regions that can actually appear on the figure.
    if SINGLE:
        union = {k: 0.0 for k in (values_left if SOURCE == "left" else values_right)}
    else:
        union = {k: 0.0 for k in set(values_left) | set(values_right)}
    union.update({k: 0.0 for k in OVERLAYS})
    hm = bgh.Heatmap(
        union,
        position=position,
        orientation=orientation,
        atlas_name=ATLAS_NAME,
        format="2D",
        vmin=0, vmax=1, cmap="Greys",
        label_regions=False, check_latest=False,
    )
    projected, _ = hm.slicer.get_structures_slice_coords(
        hm.regions_meshes, hm.scene.root
    )
    _PROJ_CACHE[key] = projected
    return projected


def _draw_slice(ax, projected):
    """Render one coronal slice, data-colored, into an existing axis."""
    root_xy = np.vstack([c for k, c in projected.items() if k.startswith("root")])
    midline = 0.5 * (root_xy[:, 0].min() + root_xy[:, 0].max())
    x0, x1  = root_xy[:, 0].min(), root_xy[:, 0].max()
    y0, y1  = root_xy[:, 1].min(), root_xy[:, 1].max()

    big = 1e6
    clip_l = Rectangle((midline - big, -big), big, 2 * big, transform=ax.transData)
    clip_r = Rectangle((midline, -big), big, 2 * big, transform=ax.transData)

    # root outline (bottom)
    for name, coords in projected.items():
        if name.startswith("root"):
            ax.fill(coords[:, 0], coords[:, 1], color="lightgray",
                    alpha=0.3, lw=0.7, ec="k", zorder=0)

    # overlays UNDER the data
    for key, coords in projected.items():
        acr = key.split("_segment_")[0]
        if acr in OVERLAYS:
            s = OVERLAYS[acr]
            ax.fill(coords[:, 0], coords[:, 1], color=s["face"], ec=s["edge"],
                    lw=s["lw"], alpha=s["alpha"], zorder=s["zorder"])

    # data ON TOP: largest polygons first so children beat parents
    ordered = sorted(
        ((k, c) for k, c in projected.items()
         if not k.startswith("root")
         and k.split("_segment_")[0] not in OVERLAYS),
        key=lambda kv: -_poly_area(kv[1]),
    )

    # ### CHOICE ###
    # Which (data, colormap, norm, clip) passes to run. With a single
    # hemisphere the clip still applies, so regions whose polygons cross the
    # midline are cut at the midline rather than spilling into the empty half.
    if SINGLE:
        _clip = clip_l if HEMISPHERE == "left" else clip_r
        passes = [(values_left if SOURCE == "left" else values_right,
                   LEFT_CMAP if SOURCE == "left" else RIGHT_CMAP,
                   LEFT_NORM if SOURCE == "left" else RIGHT_NORM,
                   _clip)]
    else:
        passes = [(values_left,  LEFT_CMAP,  LEFT_NORM,  clip_l),
                  (values_right, RIGHT_CMAP, RIGHT_NORM, clip_r)]

    Z0 = 10
    for vals, cmap, norm, clip in passes:
        for z, (key, coords) in enumerate(ordered, start=Z0):
            v = vals.get(key.split("_segment_")[0])
            color = MISSING_COLOR if v is None else cmap(norm(v))
            poly = ax.fill(coords[:, 0], coords[:, 1], color=color,
                           lw=0.25, ec="k", zorder=z)[0]
            poly.set_clip_path(clip)

    # ── Midline: divider between sources, or the cut edge of a half section ──
    if DRAW_MIDLINE:
        ax.plot([midline, midline], [y0, y1], color=MIDLINE_COLOR, lw=MIDLINE_LW,
                ls="-", alpha=MIDLINE_ALPHA, zorder=9500)

    # ── Label(s), directly above THIS panel's own slice ──────────────────────
    if LABEL_EACH_PANEL:
        label_y = y0 - LABEL_Y_PAD_UM   # y0 = dorsal-most extent (axis is inverted, "up")
        if SINGLE:
            # ### CHOICE ###
            # Axes coordinates, not data coordinates. Slices differ in size, so
            # anchoring to each slice's own dorsal extent puts the label at a
            # different height in every panel and lets the big ones overflow
            # into the row above. Axes-relative keeps one baseline for all.
            ax.text(0.5, 1.005, LEFT_LABEL if SOURCE == "left" else RIGHT_LABEL,
                    transform=ax.transAxes, ha="center", va="bottom",
                    fontsize=LABEL_FONTSIZE, fontweight="bold",
                    zorder=10000, clip_on=False)
        else:
            ax.text(0.5 * (x0 + midline), label_y, LEFT_LABEL, ha="center", va="bottom",
                    fontsize=LABEL_FONTSIZE, fontweight="bold", zorder=10000, clip_on=False)
            ax.text(0.5 * (midline + x1), label_y, RIGHT_LABEL, ha="center", va="bottom",
                    fontsize=LABEL_FONTSIZE, fontweight="bold", zorder=10000, clip_on=False)


def _build_grid_axes(positions, orientation="frontal"):
    """Shared boilerplate for both the data grid and the ontology-legend
    grid: compute one common mm-scale frame across all panels, create the
    subplot grid, and return everything needed to draw into it. Factored out
    so plot_grid() and plot_ontology_grid() don't duplicate this math."""
    n = len(positions)
    projections = {p: _get_projected(p, orientation) for p in positions}

    all_xy = np.vstack([c for proj in projections.values() for c in proj.values()])
    gx0, gx1 = all_xy[:, 0].min(), all_xy[:, 0].max()
    gy0, gy1 = all_xy[:, 1].min(), all_xy[:, 1].max()

    # ### CHOICE ###
    # One midline for the whole grid, taken from the root outline pooled across
    # every slice, so all panels are cropped identically. Computing it per-slice
    # (as _draw_slice does for its own divider) would let the crop wobble by a
    # few microns between panels and misalign the column edges.
    root_xy = np.vstack([c for proj in projections.values()
                         for k, c in proj.items() if k.startswith("root")])
    midline = 0.5 * (root_xy[:, 0].min() + root_xy[:, 0].max())

    if SINGLE:
        # Crop x to the chosen half; keep the full dorsoventral extent.
        if HEMISPHERE == "left":
            xlim = (gx0 - PANEL_PAD_UM, midline + PANEL_PAD_UM)
        else:
            xlim = (midline - PANEL_PAD_UM, gx1 + PANEL_PAD_UM)
        ylim = (gy0 - PANEL_PAD_UM, gy1 + PANEL_PAD_UM)
    else:
        cx, cy = 0.5 * (gx0 + gx1), 0.5 * (gy0 + gy1)
        half = 0.5 * max(gx1 - gx0, gy1 - gy0) + PANEL_PAD_UM
        xlim = (cx - half, cx + half)
        ylim = (cy - half, cy + half)

    # ### CHOICE ###
    # Panel proportions follow the frame instead of always being square, so a
    # half-brain isn't letterboxed inside a square axis. PANEL_SIZE now sets the
    # larger dimension and the other shrinks in proportion.
    fw, fh = xlim[1] - xlim[0], ylim[1] - ylim[0]
    scale = PANEL_SIZE / max(fw, fh)
    panel_w, panel_h = fw * scale, fh * scale

    fig, axes = plt.subplots(
        GRID_ROWS, GRID_COLS,
        figsize=(GRID_COLS * panel_w, GRID_ROWS * panel_h),
        gridspec_kw={"hspace": GRID_HSPACE},
    )
    axes = np.atleast_1d(axes).ravel()
    return fig, axes, projections, xlim, ylim, n


def _finish_grid(fig, axes, xlim, ylim, n):
    """Apply shared axis limits/orientation, hide unused panels, draw the
    scale bar. Shared tail-end of both grid-plotting functions."""
    for ax in axes[:n]:
        ax.set_xlim(*xlim)
        ax.set_ylim(ylim[1], ylim[0])      # inverted: dorsal up
        ax.set_aspect("equal")
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")

    sb_ax = axes[n - 1]
    L = SCALEBAR_MM * 1000
    span_x, span_y = xlim[1] - xlim[0], ylim[1] - ylim[0]
    x_end = xlim[1] - 0.06 * span_x
    y_bar = ylim[1] - 0.10 * span_y
    sb_ax.plot([x_end - L, x_end], [y_bar, y_bar], color="k", lw=2.5,
               solid_capstyle="butt", zorder=10000, clip_on=False)
    sb_ax.text(x_end - L / 2, y_bar - 0.03 * span_y, f"{SCALEBAR_MM:g} mm",
               ha="center", va="bottom", fontsize=8, zorder=10000, clip_on=False)


def plot_grid(positions=CORONAL_POSITIONS, orientation="frontal", save_path=None):
    fig, axes, projections, xlim, ylim, n = _build_grid_axes(positions, orientation)

    for ax, pos in zip(axes, positions):
        _draw_slice(ax, projections[pos])

    _finish_grid(fig, axes, xlim, ylim, n)

    fig.subplots_adjust(left=0.05, right=0.88, top=0.90, bottom=0.04,
                        wspace=0.02, hspace=GRID_HSPACE)

    # get the middle row's vertical extent to center the colorbar on it
    axes_grid = axes.reshape(GRID_ROWS, GRID_COLS)
    mid_row = axes_grid[GRID_ROWS // 2]  # middle row (row index 1 for GRID_ROWS==3)
    row_positions = [ax.get_position() for ax in mid_row]
    y0 = min(p.y0 for p in row_positions)
    y1 = max(p.y1 for p in row_positions)

    if SINGLE:
        # One bar, no "(left hemi)" suffix -- there is only one hemisphere and
        # only one metric on the figure, so the side carries no information.
        cmap_, norm_, lab_ = ((LEFT_CMAP, LEFT_NORM, LEFT_LABEL) if SOURCE == "left"
                              else (RIGHT_CMAP, RIGHT_NORM, RIGHT_LABEL))
        cax = fig.add_axes([0.905, y0, 0.011, y1 - y0])
        cb = fig.colorbar(ScalarMappable(norm=norm_, cmap=cmap_), cax=cax)
        cb.set_label(lab_, fontsize=9)
        cb.ax.tick_params(labelsize=8)
    else:
        cax_r = fig.add_axes([0.905, y0, 0.011, y1 - y0])
        cb_r = fig.colorbar(ScalarMappable(norm=RIGHT_NORM, cmap=RIGHT_CMAP), cax=cax_r)
        cb_r.set_label(f"{RIGHT_LABEL}  (right hemi)", fontsize=9)
        cb_r.ax.tick_params(labelsize=8)

        #cax_l = fig.add_axes([0.905, 0.10, 0.011, 0.34])
        #cb_l = fig.colorbar(ScalarMappable(norm=LEFT_NORM, cmap=LEFT_CMAP), cax=cax_l)
        #cb_l.set_label(f"{LEFT_LABEL}  (left hemi)", fontsize=9)
        #cb_l.ax.tick_params(labelsize=8)

    #fig.suptitle(f"cFos Density -- {LEFT_LABEL} (L) vs {RIGHT_LABEL} (R)",
    #             fontsize=14, fontweight="bold", y=0.97)

    if save_path and SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    plt.close(fig)


## 5. Generate the data grid

In [ ]:
_tag = (f"{LEFT_LABEL}_vs_{RIGHT_LABEL}" if not SINGLE
        else f"{LEFT_LABEL if SOURCE == 'left' else RIGHT_LABEL}_{HEMISPHERE}hemi")
plot_grid(save_path=OUTPUT_DIR / f"grid_{_tag}.png")

## 6. Ontology-color reference grid (legend)

Same coronal grid, same slice positions, but every region is filled with the
Allen atlas's **own default ontology color** (`rgb_triplet`) instead of your
data. No colorbar -- the point isn't a continuous scale, it's "what region is
this," meant to sit next to the data grid above as a visual key for anatomical
identity by position.

In [ ]:
def _ontology_color(acr):
    try:
        rgb = atlas.structures[acr]["rgb_triplet"]
        return tuple(v / 255 for v in rgb)
    except (KeyError, TypeError):
        return (0.85, 0.85, 0.85)   # fallback for anything unmapped


def _draw_slice_ontology(ax, projected):
    """Render one coronal slice filled with Allen's own per-region colors.
    No missing-data logic -- every region in the atlas gets its official color,
    since this plot isn't about your data at all. It does honour HEMISPHERE, so
    the legend grid stays geometrically aligned with the data grid beside it."""
    root_xy = np.vstack([c for k, c in projected.items() if k.startswith("root")])
    midline = 0.5 * (root_xy[:, 0].min() + root_xy[:, 0].max())

    clip = None
    if SINGLE:
        big = 1e6
        clip = (Rectangle((midline - big, -big), big, 2 * big, transform=ax.transData)
                if HEMISPHERE == "left" else
                Rectangle((midline, -big), big, 2 * big, transform=ax.transData))

    for name, coords in projected.items():
        if name.startswith("root"):
            p = ax.fill(coords[:, 0], coords[:, 1], color="white",
                        alpha=1.0, lw=0.7, ec="k", zorder=0)[0]
            if clip is not None:
                p.set_clip_path(clip)

    for key, coords in projected.items():
        acr = key.split("_segment_")[0]
        if acr.startswith("root"):
            continue
        if acr in OVERLAYS:
            s = OVERLAYS[acr]
            p = ax.fill(coords[:, 0], coords[:, 1], color=s["face"], ec=s["edge"],
                        lw=s["lw"], alpha=s["alpha"], zorder=s["zorder"])[0]
        else:
            p = ax.fill(coords[:, 0], coords[:, 1], color=_ontology_color(acr),
                        lw=0.2, ec="k", zorder=10)[0]
        if clip is not None:
            p.set_clip_path(clip)

    if DRAW_MIDLINE:
        y0, y1 = root_xy[:, 1].min(), root_xy[:, 1].max()
        ax.plot([midline, midline], [y0, y1], color=MIDLINE_COLOR, lw=MIDLINE_LW,
                ls="-", alpha=MIDLINE_ALPHA, zorder=9500)


def plot_ontology_grid(positions=CORONAL_POSITIONS, orientation="frontal", save_path=None):
    fig, axes, projections, xlim, ylim, n = _build_grid_axes(positions, orientation)

    for ax, pos in zip(axes, positions):
        _draw_slice_ontology(ax, projections[pos])

    _finish_grid(fig, axes, xlim, ylim, n)

    fig.subplots_adjust(left=0.05, right=0.97, top=0.90, bottom=0.04,
                        wspace=0.02, hspace=GRID_HSPACE)
    fig.suptitle(f"Region ontology -- Allen atlas default colors  ({ATLAS_NAME})",
                 fontsize=14, fontweight="bold", y=0.97)

    if save_path and SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    plt.close(fig)


_onto_tag = "grid_ontology_legend" if not SINGLE else f"grid_ontology_legend_{HEMISPHERE}hemi"
plot_ontology_grid(save_path=OUTPUT_DIR / f"{_onto_tag}.png")
